In [10]:
import pandas as pd
import pyodbc
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sqlalchemy import create_engine

In [4]:
engine = create_engine( "mssql+pyodbc://localhost\\SQLEXPRESS/shopeasy?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")

with engine.connect() as connection:
    print("connection successful")

connection successful


In [ ]:
query ="select * from dbo.customer_reviews_cleaned"

df_review = pd.read_sql(query,engine)


In [7]:
df_review.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,19,34,13,04-01-2024 12:00:00 AM,4,The quality is top-notch.
1,22,52,3,30-10-2025 12:00:00 AM,4,The quality is top-notch.
2,27,56,6,24-11-2025 12:00:00 AM,4,The quality is top-notch.
3,32,91,1,15-07-2023 12:00:00 AM,4,The quality is top-notch.
4,94,2,16,14-03-2025 12:00:00 AM,4,The quality is top-notch.


In [9]:
print("null count :",df_review.isna().sum())
print("duplicates : ",df_review.duplicated().sum())
print("Zero ratings:", (df_review['Rating'] == 0).sum())

null count : ReviewID      0
CustomerID    0
ProductID     0
ReviewDate    0
Rating        0
ReviewText    0
dtype: int64
duplicates :  0
Zero ratings: 0


All Validation checks Passed , No Nulls,No Duplicates,No Zero Ratings

In [19]:
analyser = SentimentIntensityAnalyzer()
df_review["SentimentScore"]= df_review['ReviewText'].apply(lambda x: analyser.polarity_scores(x)['compound'])

In [21]:
df_review[['ReviewText','SentimentScore']].head(10)

,ReviewText,SentimentScore
0,The quality is top-notch.,0.0
1,The quality is top-notch.,0.0
2,The quality is top-notch.,0.0
3,The quality is top-notch.,0.0
4,The quality is top-notch.,0.0
5,The quality is top-notch.,0.0
6,The quality is top-notch.,0.0
7,The quality is top-notch.,0.0
8,The quality is top-notch.,0.0
9,The quality is top-notch.,0.0


In [23]:
def catergory_segemnt(score,rating):
    if score>0.05:
        if rating>=4:
            return "Positive"
        elif rating ==3:
            return "Mixed Positive"
        else:
            return "Mixed Negative"
    elif score <-0.05:
        if rating <=2:
            return "Negative"
        elif rating ==3:
            return "Mixed Negative"
        else:
            return "MIxed positive"
    else:
        if rating>=4:
            return "Positve"
        elif rating<=2:
            return "Negative"
        else:
            return "Neutral"

df_review["sentimentCategory"]=df_review.apply(lambda row: catergory_segemnt(row['SentimentScore'],row["Rating"]),axis=1)


In [26]:
def bucket(score):
    if score>=0.5 :
        return "0.5 to 1.0"
    elif 0.0<= score<0.5:
        return "0.0 to 0.49"
    elif -0.5<=score<0.0:
        return "-0.49 to 0.0"
    else:
        return "-1.0 to -0.5"

df_review["SentimentBucket"] =df_review['SentimentScore'].apply(bucket)
    


In [27]:
df_review.head(5)

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,Sentiment_Score,SentimentScore,sentimentCategory,SentimentBucket
0,19,34,13,04-01-2024 12:00:00 AM,4,The quality is top-notch.,0.0,0.0,Positve,0.0 to 0.49
1,22,52,3,30-10-2025 12:00:00 AM,4,The quality is top-notch.,0.0,0.0,Positve,0.0 to 0.49
2,27,56,6,24-11-2025 12:00:00 AM,4,The quality is top-notch.,0.0,0.0,Positve,0.0 to 0.49
3,32,91,1,15-07-2023 12:00:00 AM,4,The quality is top-notch.,0.0,0.0,Positve,0.0 to 0.49
4,94,2,16,14-03-2025 12:00:00 AM,4,The quality is top-notch.,0.0,0.0,Positve,0.0 to 0.49


In [28]:
df_review.to_csv("review_sentiments.csv",index=False)

In [32]:
from collections import Counter
import re

df_negative = df_review[df_review["sentimentCategory"]=="Negative"]

negative_text = ' '.join(df_review["ReviewText"].str.lower())
words= re.findall(r'\b\w+\b',negative_text)
words_count = Counter(words)

stop_words = ['the', 'a', 'is', 'it', 'was', 'and', 'for', 'to', 'of', 'my', 'i', 'not', 'be', 'very', 'this', 'that', 'with', 'on', 'are', 'but']
filtered ={}
for w,c in words_count.items():
    if w not in stop_words:
        filtered[w]=c
print(Counter(filtered).most_common(40))

[('product', 329), ('quality', 299), ('customer', 123), ('good', 122), ('could', 122), ('cheaper', 122), ('top', 121), ('notch', 121), ('support', 114), ('helpful', 114), ('satisfied', 114), ('worth', 113), ('money', 113), ('buy', 108), ('again', 108), ('great', 105), ('purchase', 105), ('five', 104), ('stars', 104), ('quick', 104), ('delivery', 104), ('experience', 100), ('love', 99), ('will', 99), ('average', 91), ('nothing', 91), ('special', 91), ('shipping', 88), ('fast', 88), ('item', 88), ('well', 88), ('packaged', 88), ('expectations', 87), ('okay', 73), ('instructions', 73), ('were', 73), ('unclear', 73), ('build', 56), ('impressive', 56), ('excellent', 54)]


In [34]:
def categorize_issue(text):
    text = text.lower()
    if any(word in text for word in ["worth", "money", "expensive", "price", "costly"]):
        return "Product Cost"
    elif any(word in text for word in ["arrived", "late", "delivery", "shipping"]):
        return "Delivery"
    elif any(word in text for word in ["meet", "expectations", "average", "nothing", "special"]):
        return "Unmet Expectations"
    elif any(word in text for word in ["stopped", "working", "broke", "month"]):
        return "Durability"
    elif any(word in text for word in ["performance", "disappointed", "bad", "terrible"]):
        return "Product Performance"
    elif any(word in text for word in ["instructions", "unclear", "confusing"]):
        return "Usability"
    elif any(word in text for word in ["customer", "service", "support"]):
        return "Customer Service"
    else:
        return "General Dissatisfaction"
    
df_negative = df_review[df_review['sentimentCategory'] == 'Negative'].copy()

df_negative['IssueCategory'] = df_negative['ReviewText'].apply(categorize_issue)

print(df_negative['IssueCategory'].value_counts())

IssueCategory
Unmet Expectations         70
Product Cost               47
Product Performance        40
Delivery                   38
Durability                 18
Usability                   9
General Dissatisfaction     4
Name: count, dtype: int64


In [35]:
df_negative.to_csv('nega_reviews_issues.csv', index=False)